# Introduction

## Project Motivation

This project started from a simple curiosity:

> **Do major film studios benefit from a "halo effect"?**

Major film studios often have:
- Large marketing budgets
- Strong brand recognition
- Established fan bases

These advantages made me wonder whether audiences might evaluate movies differently simply because they are associated with well-known studios.

However, directly measuring a potential **halo effect** is challenging. Therefore, I use the difference between **audience ratings** and **professional critic ratings** as an indicator.

While critic ratings do not represent an absolute measure of movie quality, professional critics typically apply more systematic evaluation standards and are less directly influenced by marketing exposure or studio reputation. This makes critic ratings a useful benchmark for comparing audience responses across different types of films.

If audiences consistently rate major studio films more positively relative to critics, compared with non-major studio films, this may suggest the presence of a studio-related halo effect.

---

## Research Question

Based on this motivation, the central research question of this project is:

> **Do major studio films show larger audience–critic rating gaps than non-major studio films?**
---

## Data Sources

No single API provides all the information required for this project. Therefore, I combine data from both **TMDB** and **OMDb**, allowing each API to contribute its strengths.

### TMDB API

TMDB serves as the primary source for movie metadata and production information. It is used to:

- (Discover endpoint) Discover movies that meet the selection criteria
- (Movie details) Retrieve production companies for classifying movies as major or non-major
- Collect audience-related information, such as:
  - TMDB audience rating (`vote_average`)
  - Number of audience ratings (`vote_count`)
  - Budget and revenue
  - IMDb ID
  - ...

However, TMDB does **not** provide professional critic ratings.

### OMDb API

OMDb complements TMDB by providing information that is unavailable from the TMDB API, including:

- Professional critic rating(`Metascore`)
- IMDb audience rating(`imdbID`)
- Genre
- Release year
- ...

Each OMDb request requires an **IMDb ID**, which is obtained from the TMDB Movie Details endpoint. The IMDb ID also serves as a common identifier for merging the two datasets.

---

## Main Variables:

**Production information**
- Studio type

**Movie characteristics**
- Genre
- Release year
- Budget
- Revenue

**Audience evaluation**
- TMDB vote average
- IMDb rating
- Audience Score *(the average of TMDB and IMDb ratings)*

**Critic evaluation**
- Metascore
- Critic Score *(converting Metascore to a 10-point scale)*

**Analytical variable**
- Audience–Critic Gap *(the difference between the Audience Score and the Critic Score)*

---

## Defining Major and Independent Studios

To classify movies consistently, I adopted the definition of **major film studios** from the Wikipedia page [Major film studios](https://en.wikipedia.org/wiki/Major_film_studios). The page describes the **"Big Five"** studios as the dominant companies in the American film industry, collectively accounting for approximately **80–85% of the U.S. box office revenue**.

The five major studios used in this project are:

- Walt Disney Pictures
- Warner Bros.
- Universal Pictures
- Paramount Pictures
- Sony Pictures

Company ID Source: [tmdb_search_company](https://api.themoviedb.org/3/search/company)

For each movie, the TMDB **Movie Details** endpoint returns a list of production companies. A movie is classified as a **major** if **at least one** of its production companies belongs to the Big Five listed above. Otherwise, it is classified as non-major (referred to as **"indie"** throughout this project)*.

This rule provides a simple, transparent way to distinguish between major studio and non-major productions throughout the project.

In [ ]:
MAJOR_COMPANY_IDS = {2,4,33,34,174}  # Disney, Warner Bros, Universal, Paramount, Sony

## Ensuring a Fair Comparison

To make the comparison between major and non-major films more meaningful, I applied the same selection criteria to both groups.

**1. Same release period**

Only movies released between **2015 and 2024** were included. Restricting the analysis to the same ten-year period helps control for changes in the film industry, audience behaviour, and rating platforms over time.

**2. Minimum audience engagement**

Only movies with a TMDB `vote_count` of at least **2,000** were selected. This ensures that audience ratings are based on a sufficiently large number of users, reducing the influence of unstable ratings from films with very limited audience responses.

**3. Consistent data collection process**

Major and non-major films were collected using the same API endpoints, filtering conditions, and data processing procedures. This ensures that differences observed between the two groups are less likely to result from inconsistent data collection methods.

---

# Project Overview

This project follows a complete data engineering workflow, from API data collection to data processing, transforminng, and exploratory analysis.

```text
NB01a ───────────────────────────────────────────────
TMDB API Collection
│
├─ Fetch movie list (tmdb-Discover endpoint)
├─ Fetch detailed movie information(tmdb-Movie Details endpoint)
│
└──► raw/
      ├── tmdb_major.json
      └── tmdb_indie.json


NB01b ───────────────────────────────────────────────
OMDb API Collection
(using IMDb IDs obtained from NB01a)
│
├─ Fetch movie information(omdb-By ID endpoint)
│
└──► raw/
      ├── omdb_major.json
      └── omdb_indie.json


NB02a ───────────────────────────────────────────────
TMDB Data Processing
│
├─ JSON → DataFrame
├─ Data cleaning
├─ Select variables
└──► data/processed/tmdb_movies.csv


NB02b ───────────────────────────────────────────────
OMDb Data Processing
│
├─ JSON → DataFrame
├─ Data cleaning
├─ Select variables
└──► data/processed/omdb_movies.csv
│
├─ Merge with processed TMDB dataset
│    (using IMDb ID)
│
├─ Feature Engineering
│    • audience_score
│    • metascore_10
│    • gap
│
└──► movies_clean.csv


NB03 ───────────────────────────────────────────────
Exploratory Data Analysis
│
├─ Overall Comparison
├─ Distribution Exploration
├─ Genre Analysis
├─ Further Exploration
│
├─ Interpretation
├─ Limitations
└─ Conclusion
```

## Setup

In [1]:
import requests
import json
import os
from dotenv import load_dotenv

## Loading API key

In [2]:
load_dotenv()
TMDB_API_KEY = os.getenv("TMDB_API_KEY")

if TMDB_API_KEY:
    print("✅ TMDB API key loaded successfully!")
else:
    print("❌ TMDB API key not found.")


✅ TMDB API key loaded successfully!


# Step 1. Discover Movies List

The first step is to retrieve a list of candidate movies using TMDB's **`discover/movie`** endpoint. Rather than searching for movies by title, the Discover endpoint allows me to apply a consistent set of filtering criteria and collect movies that satisfy the project's requirements.

The following query parameters are used:

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `vote_count.gte` | `2000` | Include only movies with a sufficiently large number of audience ratings to improve rating reliability. |
| `primary_release_date.gte` | `2015-01-01` | Restrict the dataset to recent movies. |
| `primary_release_date.lte` | `2024-12-31` | Create a consistent 10-year comparison period. |
| `sort_by` | `vote_count.desc` | This does not affect the final sample because all available result pages are collected. |
| `page` | `page_number` | Iterate through all result pages returned by the API. |

In [3]:

BASE_URL = "https://api.themoviedb.org/3"


def fetch_discover_page(page_number):
    """
    Fetch a single page of results from TMDB's discover/movie endpoint,
    filtered by vote count and release year range.
    """
    params = {
        "api_key": TMDB_API_KEY,
        "vote_count.gte": 2000,
        "primary_release_date.gte": "2015-01-01",
        "primary_release_date.lte": "2024-12-31",
        "sort_by": "vote_count.desc",
        "page": page_number
    }
    response = requests.get(f"{BASE_URL}/discover/movie",params=params)
    if response.status_code != 200:
        print("Request wasnt successful!")
        return None
    return response.json()

##### Notes: Why function?
The API returns results in multiple pages rather than a single response. I encapsulate the request into the `fetch_discover_page()` function so that different pages can be retrieved simply by changing the page number.

### Checking Available Results and Pagination

After defining the API request function, I first retrieve the first page of results to inspect the overall size of the dataset.


In [4]:
fetch_discover_page(1)

{'page': 1,
 'results': [{'adult': False,
   'backdrop_path': '/en971MEXui9diirXlogOrPKmsEn.jpg',
   'genre_ids': [28, 12, 35],
   'id': 293660,
   'title': 'Deadpool',
   'original_language': 'en',
   'original_title': 'Deadpool',
   'overview': 'The origin story of former Special Forces operative turned mercenary Wade Wilson, who, after being subjected to a rogue experiment that leaves him with accelerated healing powers, adopts the alter ego Deadpool. Armed with his new abilities and a dark, twisted sense of humor, Deadpool hunts down the man who nearly destroyed his life.',
   'popularity': 26.4717,
   'poster_path': '/3E53WEZJqP6aM84D8CckXx4pIHw.jpg',
   'release_date': '2016-02-09',
   'softcore': False,
   'video': False,
   'vote_average': 7.624,
   'vote_count': 32977},
  {'adult': False,
   'backdrop_path': '/mDfJG3LC3Dqb67AZ52x3Z0jU0uB.jpg',
   'genre_ids': [12, 28, 878],
   'id': 299536,
   'title': 'Avengers: Infinity War',
   'original_language': 'en',
   'original_title'

### Inspecting TMDB API Response Structure

Before collecting all movie IDs, I inspected the structure of the API response to understand the returned data format.


In [5]:
fetch_discover_page(1).keys()

dict_keys(['page', 'results', 'total_pages', 'total_results'])

In [ ]:
type(fetch_discover_page(1)["results"])

list

In [8]:
len(fetch_discover_page(1)["results"])

20

In [6]:
first_page = fetch_discover_page(1)
total_pages = first_page["total_pages"]
print("Total pages:", total_pages)
print("Total results:", first_page["total_results"])

Total pages: 44
Total results: 877


### Extracting Movie IDs

From the Discover API response, I only extract the **movie IDs** from each result.

Although the Discover endpoint already provides some basic movie information, I do not store these fields because the TMDB **Movie Details** endpoint provides the same information along with additional variables required for the analysis, such as production companies, budget, revenue, and ratings.

Therefore, only `movie_id` values are retained at this stage. These IDs are used in the next step to retrieve detailed movie information for each candidate movie.

In [7]:
TMDB_ids =[]

for i in range(1,total_pages+1):
    page_info = fetch_discover_page(i)
    for movie in page_info["results"]:
        movie_id = movie["id"]
        TMDB_ids.append(movie_id)

print(len(TMDB_ids))

877


## Step 2. Retrieve Detailed Movie Information

The `movie_id` from the Discover response is used as the unique identifier to request detailed information for individual movies.

The Movie Details endpoint provides additional information including:

- Production companies
- Budget
- Revenue
- Audience ratings
- Vote count
- Release information
- IMDb ID

Among these variables, **production companies** are especially important because they are used to classify movies into major and non-major groups.

In [ ]:
def fetch_movie_detail(movie_id):
    """
    Fetch full details for a single movie from TMDB.
    """
    params = {
        "api_key": TMDB_API_KEY,
    }
    response = requests.get(f"{BASE_URL}/movie/{movie_id}", params=params)
    if response.status_code != 200:
        print("request wasnt successful!")
        return None
    return response.json()

In [ ]:
test = fetch_movie_detail(293660)

In [ ]:
type(test)
test.keys() #Too many unnecessary keys in the result

dict_keys(['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count'])

The TMDB Movie Details response contains many additional fields, but storing the complete response would introduce unnecessary information and make later processing more complicated. Therefore, I define a list of relevant fields and extract only the required variables.

| Field | Decision | Reason |
|-------|----------|--------|
| `id` | ✅ Keep | Unique TMDB movie identifier. |
| `imdb_id` | ✅ Keep | Required to retrieve additional ratings from the OMDb API. |
| `title` | ✅ Keep | Useful for identification and interpretation. |
| `release_date` | ✅ Keep | Used to identify the movie release year, could be useful in trend analysis |
| `production_companies` | ✅ Keep | Used to classify movies as major-studio or non-major ones. |
| `budget` | ✅ Keep | Potential explanatory variable for later analysis. |
| `revenue` | ✅ Keep | Potential explanatory variable for later analysis. |
| `vote_average` | ✅ Keep | TMDB user rating. |
| `vote_count` | ✅ Keep | Used as a popularity threshold during data collection. |
| `genres` | ⏳ Optional | May be useful for exploratory analysis,but omdb provide a non-nested output so leave it out for now. |
| `runtime` | ❌ Exclude | Irrelevant to the research question. |
| `original_language` | ❌ Exclude | Irrelevant to the research question. |
| `overview` | ❌ Exclude | Long text description.Irrelevant to the research question. |
| `tagline` | ❌ Exclude | Marketing text; not required for analysis. |
| `homepage` | ❌ Exclude | External website link; not relevant. |
| `poster_path` | ❌ Exclude | Image file path; not required. |
| `backdrop_path` | ❌ Exclude | Image file path; not required. |
| `belongs_to_collection` | ❌ Exclude | Franchise information; outside the scope of this project. |
| `spoken_languages` | ❌ Exclude | Not relevant to the research question. |
| `production_countries` | ❌ Exclude | Not required for this analysis. |
| `status` | ❌ Exclude | Most selected movies are already released; this field provides little additional information. |
| `adult` | ❌ Exclude | Not relevant after the initial movie selection. |
| `video` | ❌ Exclude | Indicates whether a record is a video release; not relevant for this project. |

Based on this inspection, only the fields required for movie identification, studio classification, API matching, and subsequent analysis were retained during data collection. This reduces unnecessary data storage while ensuring that all variables needed for the project are available.

### Fetching Movie Details

Using the collected `movie_id` list from the Discover endpoint, I now retrieve detailed information for each movie through the TMDB Movie Details endpoint.

In [ ]:
all_movie_details=[]
Keep_Fields = [
    "id",
    "title",
    "imdb_id",
    "release_date",
    "production_companies",
    "budget",
    "revenue",
    "vote_average",
    "vote_count"
]

for movie_id in TMDB_ids:
    details = fetch_movie_detail(movie_id)
    selected_movie_detail = {
        field: details[field] for field in Keep_Fields
    }
    all_movie_details.append(selected_movie_detail)


### Inspect the returned results

In [12]:
type(all_movie_details[0])
all_movie_details[0].keys()

dict_keys(['id', 'title', 'imdb_id', 'release_date', 'production_companies', 'budget', 'revenue', 'vote_average', 'vote_count'])

In [ ]:
all_movie_details[0]

### Validating IMDb IDs

Before collecting additional information from the OMDb API, I check whether all movies have a valid `imdb_id`.

The IMDb ID is required for OMDb API requests and also serves as the common identifier for matching TMDB and OMDb records later in the pipeline. Therefore, missing IMDb IDs would prevent some movies from being collected or merged correctly.

This validation step counts the number of movies without an IMDb ID to ensure that the dataset is ready for the next data collection stage.

In [ ]:
missing_imdb_id = 0
# Use .get() to safely access the "imdb_id" field.
# If the key does not exist, it returns None instead of causing an error.
for movie in all_movie_details:
    if movie.get("imdb_id") is None:
        missing_imdb_id += 1

print("Missing IMDb IDs:", missing_imdb_id)

Missing IMDb IDs: 0


## Step 3: Classifying Movies by Studio Type

Using the `production_companies` information from TMDB Movie Details, movies are classified into two groups.

For each movie, the production company IDs are compared with the predefined `MAJOR_COMPANY_IDS`:
- If any production company matches a major studio, the movie is labelled as **major**.
- Otherwise, it is labelled as **non-major (indie)**.

The two groups are stored separately as:
- `major_movies`
- `indie_movies`

In [15]:
major_movies = []
indie_movies = []

for movie in all_movie_details:
    companies = movie["production_companies"]
    is_major = False
    for company in companies:
        company_id = company["id"]
        if company_id in MAJOR_COMPANY_IDS:
            is_major = True
            break

    if is_major:
        major_movies.append(movie)

    else:
        indie_movies.append(movie)

In [16]:
print(major_movies)
print(indie_movies)

[{'id': 475557, 'title': 'Joker', 'imdb_id': 'tt7286456', 'release_date': '2019-10-01', 'production_companies': [{'id': 174, 'logo_path': '/zhD3hhtKB5qyv7ZeL4uLpNxgMVU.png', 'name': 'Warner Bros. Pictures', 'origin_country': 'US'}, {'id': 83036, 'logo_path': None, 'name': 'Joint Effort', 'origin_country': 'US'}, {'id': 79, 'logo_path': '/at4uYdwAAgNRKhZuuFX8ShKSybw.png', 'name': 'Village Roadshow Pictures', 'origin_country': 'US'}, {'id': 13240, 'logo_path': '/aTc07YaNHo8WNgkQSnvLmG6w4nW.png', 'name': 'Bron Studios', 'origin_country': 'CA'}, {'id': 128064, 'logo_path': '/13F3Jf7EFAcREU0xzZqJnVnyGXu.png', 'name': 'DC Films', 'origin_country': 'US'}], 'budget': 55000000, 'revenue': 1078958629, 'vote_average': 8.122, 'vote_count': 27972}, {'id': 76341, 'title': 'Mad Max: Fury Road', 'imdb_id': 'tt1392190', 'release_date': '2015-05-13', 'production_companies': [{'id': 174, 'logo_path': '/zhD3hhtKB5qyv7ZeL4uLpNxgMVU.png', 'name': 'Warner Bros. Pictures', 'origin_country': 'US'}, {'id': 79, 

In [17]:
print(len(major_movies))
print(len(indie_movies))


209
668


## Step 4: Save raw data to json files

In [27]:
with open("../data/tmdb_major_movies.json","w") as f:
        json.dump(major_movies,f,indent=2)
with open("../data/tmdb_indie_movies.json","w") as f:
        json.dump(indie_movies,f,indent=2)